# SLD evaluation: Later-time reconstruction

Evaluate the supplied CaseC_1 checkpoint for each PDE at later LR starts **225, 250**. The `_1` filenames identify a representative run. Use the PDE-specific partitions from the training notebooks, as confirmed by the authors:

| PDE | Partition seed | Test ICs |
| --- | --- | --- |
| Gray–Scott | 4 | 1, 3, 7, 8, 9 |
| Ginzburg–Landau | 48 | 1, 2, 4, 7, 8 |
| Burgers | 1 | 6, 7, 8, 9, 10 |

One checkpoint is evaluated per PDE. Report pooled relative L2 and MAE over all 100 target frames, both channels and all configured windows. Small per-IC/per-window summaries are saved as JSON. Normalization is restored from each checkpoint; no training data is required to recompute it.

Gray–Scott preserves `LR[:,50::2]` and `HR[:,200::2]` before applying the starts. Both Burgers tasks use the `sig040_indIC` dataset. See [evaluation details](../docs/EVALUATION.md).

In [ ]:
from pathlib import Path
import os
import sys

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'sld' / 'evaluation.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Start Jupyter in the SLD repository or a subfolder.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from sld.evaluation import PDE_CONFIG, evaluate

TASK = 'task2_later_time'
DATA_ROOT = Path(os.environ.get('SLD_DATA_ROOT', str(ROOT / 'data'))).expanduser()
RESULTS_ROOT = Path(os.environ.get('SLD_RESULTS_ROOT', str(ROOT / 'results'))).expanduser()
DEVICE = None  # Auto: CUDA if available, otherwise CPU; or set 'cuda' / 'cpu'.
PDES = ['gray_scott', 'ginzburg_landau', 'burgers']  # Select the PDEs to evaluate.
TEST_ICS = None  # Optional dict, e.g. {'burgers': [1]}; None uses the documented splits.
DOWNLOAD_DATA = False  # True downloads only the configured test trajectories.

CHECKPOINTS = {
    'gray_scott': ROOT / TASK / 'Gray-Scott' / 'checkpoints' / 'GS_CaseC_1_best_model.pth',
    'ginzburg_landau': ROOT / TASK / 'Ginzburg-Landau' / 'checkpoints' / 'GL_CaseC_1_best_model.pth',
    'burgers': ROOT / TASK / 'Burgers' / 'checkpoints' / 'Burgers_CaseC_1_best_model.pth',
}
print('Data:', DATA_ROOT)
print('Results:', RESULTS_ROOT / TASK)


## Data

If your MATLAB files are already under `DATA_ROOT/gray_scott`, `DATA_ROOT/ginzburg_landau` and `DATA_ROOT/burgers`, leave `DOWNLOAD_DATA=False`. Otherwise set it to `True` above. The optional download reads the public [SLD dataset](https://huggingface.co/datasets/ChenhaoSI/SLD-dataset) at a fixed revision. Large HR files require several GB of RAM per trajectory; evaluation loads one identity at a time.

In [ ]:
if DOWNLOAD_DATA:
    from huggingface_hub import snapshot_download

    required = []
    for pde in PDES:
        cfg = PDE_CONFIG[pde]
        defaults = [1] if TASK == 'task1_unseen_ic' else cfg['task2_test_ics']
        ids = TEST_ICS.get(pde, defaults) if TEST_ICS is not None else defaults
        for ic in ids:
            required.extend(f"{pde}/{cfg[key].format(ic=ic)}" for key in ['lr_name', 'hr_name'])
    snapshot_download(
        repo_id='ChenhaoSI/SLD-dataset', repo_type='dataset',
        revision='99e5242aaf83fe64071b6bf6d8e6505e75165d92',
        allow_patterns=required, local_dir=str(DATA_ROOT),
    )


## Evaluate

This runs inference on the selected test trajectories. It loads the one explicit checkpoint named in `CHECKPOINTS` for each PDE. `best_error` inside the checkpoint is recorded validation metadata; the test metrics below are computed from predictions and ground truth.

In [ ]:
results = {}
for pde in PDES:
    ids = TEST_ICS.get(pde) if TEST_ICS is not None else None
    print(f'\nEvaluating {pde} ...')
    results[pde] = evaluate(
        pde=pde, task=TASK, data_root=DATA_ROOT,
        checkpoint=CHECKPOINTS[pde], device=DEVICE, test_ics=ids,
        output_path=RESULTS_ROOT / TASK / pde / 'metrics.json',
    )
    r = results[pde]
    print('Test ICs:', r['configuration']['test_ics'])
    print('LR starts:', r['configuration']['lr_starts'])
    print(f"Relative L2: {r['metrics']['relative_l2']:.6e} | MAE: {r['metrics']['mae']:.6e}")


In [ ]:
print(f"{'PDE':<20} {'Relative L2':>14} {'MAE':>14}")
for pde, result in results.items():
    m = result['metrics']
    print(f"{pde:<20} {m['relative_l2']:>14.6e} {m['mae']:>14.6e}")
